# Notebook 08 — Packaging & Submission: arm L = resep F dari stage 1 + TTS Central Javanese

Skor test terbaik: **J 0,2655** (submission 3376) ≈ **F 0,2660** (submission 2914). F = E4-native (stage 1, 1 epoch) + tahap 2 di dev resmi (NB 10 `all`) + beam 5; J = sama dengan stage 1 dua epoch. K (Homostoria) 0,2698, P 0,2664, D 0,2901, H 0,3088.

**Arm L mengubah satu variabel dari F: data stage 1 ditambah 8,02 jam TTS Central Javanese (NB 14, arm E1 jv-mix-16k).** Tahap 2 (NB 10 `tts-all`), beam, `main.py`, dan `postprocess.py` sama persis dengan F. K (tambah percakapan Indonesia) memburuk di test; L menguji arah sebaliknya, sisi Jawa. Stage 1 NB 14 seri dengan E4-native di dev (0,2339 vs 0,2317) dengan wer_jv 0,2425 → 0,2065. **Dev tidak bisa meramal perubahan stage 1; slot test yang memutuskan.**

| Arm | Checkpoint | Decode | postprocess.py | ZIP |
|---|---|---|---|---|
| **L** | NB 10 tts-all (`…-finetune-tts-all@a19346c`) | `num_beams=5` | dasar | `submission_L_tts_all_beam5.zip` |

**Tata letak file sama dengan F** (transformers 5, `processor_config.json`, 8 entri ZIP).

**Paritas:** WER dev L lewat `main.py` harus sama dengan NB 10 `tts-all`: **2.402 error** (0,137375). Itu fit data latih, bukan perkiraan test. Skor smoke platform juga tidak berarti (kelima klip smoke adalah klip dev); referensi smoke lokal L = 0,091304.

**Setting Kaggle:** GPU T4, **Internet ON** sepanjang run, secret `HF_TOKEN` aktif di notebook ini, attach `indonesian-dev`. **Waktu:** ± 15–20 menit.

**Hasil:** `/kaggle/working/submission_L_tts_all_beam5.zip` (± 1,6 GB) dan `findings/08-packaging-tts-all/` + blok `CLAUDE-HANDOFF-JSON`.


In [ ]:
# ============================================================
# CELL 1 - KNOBS, dan pin transformers SEBELUM impor apa pun
# ============================================================
# Runtime resmi memaksa `transformers<5` (runtime/pyproject.toml,
# override-dependencies). E4 dan NB 04 dilatih, disimpan, dan di-decode di
# transformers 5.0.0. Seluruh notebook ini dijalankan di versi yang sama dengan
# container, dan pin-nya harus terjadi sebelum transformers pernah diimpor sekali pun.
PIN_TRANSFORMERS = True      # False = pakai versi bawaan Kaggle (uji pembanding)
SHIP_FP16 = True             # bobot dikirim fp16: ZIP ~1,6 GB, bukan 3,2 GB. Angka tidak
                             # berubah - main.py (dan NB 04/06) memang men-decode di fp16.
RUN_NOTE  = "tts-all"        # keluaran ke findings/08-packaging-tts-all; run sebelumnya tidak tertimpa
N_DEV     = 0                # 0 = seluruh 372 klip. Angka kecil = putaran cepat, tapi
                             # cek WER vs run tersimpan hanya berarti di 372 klip penuh.
DO_EDGE   = True             # klip senyap / pendek / >30 s / rusak + run rusak total
SMOKE_LIMIT_S  = 60          # limit smoke test resmi
SMOKE_MARGIN_S = 40          # di atas ini: muat, tapi terlalu mepet (Kaggle != container)
NOTEBOOK_ID = "08-packaging"

# Checkpoint yang dipakai arm-arm di bawah. sha16 = sha256[:16] model.safetensors
# fp32 seperti yang dipush, dicatat oleh notebook pelatihnya (checkpoint_manifest.json).
# Kalau berbeda, notebook berhenti di sel 3.
CHECKPOINTS = {
    "E4":   dict(repo="amozaurus/mozilla-jv-06-data-mixing-E4",
                 rev="c79347a54d8523f9876cfbf17bd7ea217d1db800", sha16="c58b0538dc4c2008"),
    "NB04": dict(repo="amozaurus/mozilla-jv-04-whisper-core",
                 rev="9c00033f3ea9682517a1bd901a53814489b1d1ff", sha16="3fc3a15f80cb7084"),
    # NB 06 E4-native, 25 Sep 2026: step 421 (final, tanpa seleksi dev).
    "E4N":  dict(repo="amozaurus/mozilla-jv-06-data-mixing-E4-native",
                 rev="3437f4424cdee0ce3113f7fe2ccbc32e5350be7e", sha16="b3438f32cf61ebbe"),
    # NB 10 varian "all", 28 Sep 2026: E4-native + tahap 2 di 372 klip dev (335 <= 30 s), 63 step.
    "SALL": dict(repo="amozaurus/mozilla-jv-10-dev-stage-finetune-all",
                 rev="e4df20fb882531a2d8dfa72644c33bedf8c4c6eb", sha16="058f6d26fd621170"),
    # NB 10 varian "c5" (latih convo 5 saja; convo 2 held-out -0,0267). Cadangan, tidak dipaketkan.
    "SC5":  dict(repo="amozaurus/mozilla-jv-10-dev-stage-finetune-c5",
                 rev="f7db407f4e65c319d98b49d095b2ce1184840665", sha16="10dbd350fac7465b"),
    # NB 10 varian "t0-all", 30 Sep 2026: turbo zero-shot + tahap 2 resep F (372 klip dev, 335 <= 30 s),
    # 63 step. File non-bobot = milik openai/whisper-large-v3-turbo @41f01f3 (tata letak transformers 4).
    "T0ALL": dict(repo="amozaurus/mozilla-jv-10-dev-stage-finetune-t0-all",
                  rev="3f46318c5476e698bc4c8a9d196d17f3d8207396", sha16="07ce20cd6c5b887c"),
    # NB 10 varian "2ep-all", 1 Okt 2026: stage 1 E4-native 2 epoch (NB 12 run 2) + tahap 2 resep F,
    # 63 step. Tata letak file = D/F (transformers 5). (Repo T0ALL di atas sudah dihapus 1 Okt.)
    "S2EP": dict(repo="amozaurus/mozilla-jv-10-dev-stage-finetune-2ep-all",
                 rev="b368a2ce17e39d05711aa350ac60d4f45ddb400d", sha16="6680ef3cc3dab73e"),
    # NB 10 varian "homo-all", 2 Okt 2026: stage 1 E4-native + Podcast Homostoria (NB 13) + tahap 2
    # resep F, 63 step. Tata letak file = D/F (transformers 5).
    "SHOMO": dict(repo="amozaurus/mozilla-jv-10-dev-stage-finetune-homo-all",
                  rev="f93454bf41d78a393a5bc2a3c6a3fee27ad5ad10", sha16="09ad57361da47cc7"),
    # NB 10 varian "tts-all", 3 Okt 2026: stage 1 E4-native + TTS Central Javanese (NB 14) + tahap 2
    # resep F, 63 step. Tata letak file = D/F (transformers 5).
    "STTS": dict(repo="amozaurus/mozilla-jv-10-dev-stage-finetune-tts-all",
                 rev="a19346cac56f3b8691b4e4bd233e681a819746db", sha16="b607116568cfb2e7"),
}

# Satu arm = satu checkpoint + satu nilai NUM_BEAMS di main.py; hanya itu yang boleh beda.
#   ship=True  -> ZIP di /kaggle/working, untuk diunggah.
#   parity     -> run tersimpan yang HARUS disamai (kunci di REFS). None = diukur, bukan dicocokkan.
# Arm A (NB04-native) dan B (E4 beam5) sudah dikirim 24 Sep (test 0,3027 / 0,2950);
# ZIP B masih bisa dibangun ulang dengan dict(ckpt="E4", num_beams=5, ship=True, parity=None).
ARMS = {
    # H = resep F dengan titik awal turbo zero-shot (NB 10 t0-all), beam 5, main.py dan
    # postprocess.py dasar persis seperti F. Satu variabel vs F (test 0,2660, submission 2914).
    # J = resep F dengan stage 1 dua epoch (NB 12 run 2 -> NB 10 2ep-all), beam 5, main.py dan
    # postprocess.py dasar persis seperti F. Satu variabel vs F (test 0,2660, submission 2914).
    # K = resep F dengan stage 1 E4-native + Homostoria (NB 13 -> NB 10 homo-all), beam 5, main.py dan
    # postprocess.py dasar persis seperti F. Satu variabel vs F (test 0,2660): data stage 1.
    # L = resep F dengan stage 1 E4-native + TTS Central Javanese (NB 14 -> NB 10 tts-all), beam 5,
    # main.py dan postprocess.py dasar persis seperti F. Satu variabel vs F (test 0,2660): data stage 1.
    "L": dict(ckpt="STTS", num_beams=5, ship=True, parity="STTS",
              zip="submission_L_tts_all_beam5.zip",
              what="stage 1 E4-native + TTS Central Javanese + tahap 2 (resep F), beam 5"),
}
# K (2 Okt, test 0,2698, submission 3473): dict(ckpt="SHOMO", num_beams=5, ship=True, parity="SHOMO",
#   zip="submission_K_homo_all_beam5.zip", what=...).
# J (1 Okt, test 0,2655, submission 3376): dict(ckpt="S2EP", num_beams=5, ship=True, parity="S2EP",
#   zip="submission_J_2ep_all_beam5.zip", what=...).
# H (30 Sep, test 0,3088, submission 3159): dict(ckpt="T0ALL", num_beams=5, ship=True, parity="T0ALL", ...).
# P (29 Sep, test 0,2664, submission 3097) bisa dibangun ulang dengan
# dict(ckpt="SALL", num_beams=5, ship=True, parity="SALLP", rewrite="F", zip=..., what=...).
# F sendiri (28 Sep, sudah dikirim) bisa dibangun ulang dengan
# dict(ckpt="SALL", num_beams=5, ship=True, parity="SALL", zip=..., what=...).
# Kontrol N dan C (E4-native / E4 greedy) tidak dijalankan: F sendiri sudah dicocokkan dengan run
# tersimpan, dan NB 10 sudah membuktikan main.py ini mereproduksi D persis (3.965/3.965 error).
# Arm D 26 Sep bisa dibangun ulang dengan dict(ckpt="E4N", num_beams=5, ship=True, parity=None).
RUN_CONTROL = False
if not RUN_CONTROL:
    ARMS.pop("C", None)

# Pembanding, dihitung di laptop dari prediksi dev yang disimpan notebook pelatihnya
# (NB 06: findings/06-data-mixing-E4/, NB 04: findings/04-whisper-core/), dengan
# normalizer resmi + Levenshtein yang persis sama dengan sel 8. Metode yang sama
# mereproduksi angka E4 di run 22 Sep sampai digit terakhir. Jalur decode hanya
# bergantung pada durasi audio, jadi sama untuk semua checkpoint.
REFS = {
    "E4":   dict(src="NB 06", wer=0.230941, n_err=4038, clean=0.233068, smoke=0.204348),
    "NB04": dict(src="NB 04", wer=0.264684, n_err=4628, clean=0.266022, smoke=0.252174),
    # Dihitung 26 Sep dari findings/06-data-mixing-E4-native/predictions_official_dev.csv,
    # metode yang sama (dan pada hari yang sama mereproduksi baris E4 di atas persis).
    "E4N":  dict(src="NB 06 E4-native", wer=0.231341, n_err=4045, clean=0.228758, smoke=0.204348),
    # Dihitung 28 Sep dari findings/10-dev-stage-finetune-all/submission_dev_S.csv (BEAM 5, lewat
    # main.py). Ini fit data latih - checkpoint ini dilatih di 372 klip ini - bukan perkiraan test.
    "SALL": dict(src="NB 10 all (beam 5)", wer=0.140006, n_err=2448, clean=0.141201, smoke=0.104348),
    "SC5":  dict(src="NB 10 c5 (beam 5)", wer=0.152817, n_err=2672, clean=0.158349, smoke=0.1),
    # 29 Sep: prediksi dev F (NB 10 all) setelah REWRITE_TABLES["F"] diterapkan di laptop, dengan
    # postprocess.py yang dibangun sel 4 dari kode yang sama. Fit data latih, bukan perkiraan test.
    "SALLP": dict(src="NB 10 all + rewrite F", wer=0.136631, n_err=2389, clean=0.137848, smoke=0.104348),
    # 30 Sep: dari findings/10-dev-stage-finetune-t0-all/submission_dev_S.csv (beam 5, lewat main.py),
    # metode yang sama; pada hari yang sama mereproduksi baris SALL di atas persis. Fit data latih.
    "T0ALL": dict(src="NB 10 t0-all (beam 5)", wer=0.146411, n_err=2560, clean=0.147428, smoke=0.108696),
    # 1 Okt: dari findings/10-dev-stage-finetune-2ep-all/submission_dev_S.csv, metode yang sama (hari itu juga
    # mereproduksi baris SALL persis). Fit data latih.
    "S2EP": dict(src="NB 10 2ep-all (beam 5)", wer=0.140349, n_err=2454, clean=0.13986, smoke=0.086957),
    # 2 Okt: dari findings/10-dev-stage-finetune-homo-all/submission_dev_S.csv, metode yang sama (hari itu
    # juga mereproduksi baris S2EP persis). Fit data latih.
    "SHOMO": dict(src="NB 10 homo-all (beam 5)", wer=0.140063, n_err=2449, clean=0.140914, smoke=0.104348),
    # 3 Okt: dari findings/10-dev-stage-finetune-tts-all/submission_dev_S.csv, metode yang sama (hari itu
    # juga mereproduksi baris SHOMO persis). Fit data latih.
    "STTS": dict(src="NB 10 tts-all (beam 5)", wer=0.137375, n_err=2402, clean=0.138902, smoke=0.091304),
}

# Tabel rewrite per arm (kunci "rewrite" di ARMS). Token -> token, di ruang token normalizer resmi.
# "F": dipelajari 29 Sep dari prediksi F sendiri di 372 klip dev: sebuah token hipotesis diganti hanya
# kalau ejaan referensi yang sama menggantikannya >= 2 kali dan lebih sering daripada token itu benar,
# dan pasangannya beda kapital / tanda baca / varian ejaan saja. Analog held-out (c5: belajar di
# convo 5, uji di convo 2): -0,0039, CI95 [-0,0071, -0,0014]. reports/nb10-dev-stage-finetune-results.md.
REWRITE_TABLES = {"F": {'Candi': 'candi', 'Kerajaan': 'kerajaan', 'Kudu': 'kudu', 'Purba': 'purba', 'Sejarah': 'sejarah', 'Soto': 'soto', 'alhamdulilah': 'alhamdulillah', 'apa...': 'Apa', 'biyen': 'mbiyen', 'bu': 'Bu', 'dan...': 'dan', 'der': 'der...', 'dhewe': 'dewe', 'dia...': 'dia', 'enggak...': 'enggak', 'entis': 'Entis', 'kan...': 'kan', 'kidul': 'kidul...', 'kuliahe': 'kuliahé', 'mbiyen': 'biyen', 'me...': 'me', 'nduwe': 'duwe', 'nggak': 'gak', 'rada': 'rodok', 'sangat...': 'sangat', 'soko': 'saka', 'thok': 'tok', 'tok': 'thok', 'umik': 'Umi', 'umiku': 'Umiku'}}
# Skor yang sudah diketahui, hanya untuk dicetak berdampingan (bukan cek).
KNOWN = dict(F_devstage_all_beam5=dict(dev=0.140006, dev_err=2448, test=0.2660, id=2914),
             J_2ep_all_beam5=dict(dev=0.140349, dev_err=2454, test=0.2655, id=3376),
             K_homo_all_beam5=dict(dev=0.140063, dev_err=2449, test=0.2698, id=3473),
             P_devstage_all_beam5_rewrite=dict(dev=0.136631, dev_err=2389, test=0.2664, id=3097),
             H_t0_all_beam5=dict(dev=0.146411, dev_err=2560, test=0.3088, id=3159),
             D_e4native_beam5=dict(dev=0.226766, dev_err=3965, test=0.2901, id=2693),
             B_e4beam5=dict(dev=0.225965, dev_err=3951, test=0.2950, id=2537),
             E4_greedy=dict(dev=0.230941, test=0.3005, id=2507))
REF_PATHS  = dict(short=328, chunked=44, windows=88)
PARITY_TOL = 0.005
BEAM_GUARD = 0.03            # arm beam lebih buruk dari greedy checkpoint-nya lebih dari ini = beam rusak, bukan kalah
N_BOOT     = 5000            # resample untuk bootstrap berpasangan vs C

import subprocess, sys
if PIN_TRANSFORMERS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers<5"],
                       capture_output=True, text=True)
    print("pip transformers<5 ->", "OK" if r.returncode == 0 else "GAGAL")
    if r.returncode: print(r.stderr[-1500:])

import transformers, torch, numpy, platform
print("python       :", platform.python_version(), " (container: 3.12)")
print("transformers :", transformers.__version__, " (container: <5)")
print("torch        :", torch.__version__, " (container: 2.11.0+cu130 - tidak bisa disamakan di Kaggle)")
print("numpy        :", numpy.__version__)
print("gpu          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")
MAJOR = int(transformers.__version__.split(".")[0])
if MAJOR >= 5:
    print("\n!!! transformers %s >= 5. Container memakai <5." % transformers.__version__)
    print("!!! Angka dari notebook ini TIDAK mewakili perilaku container.")
if not torch.cuda.is_available():
    raise RuntimeError("GPU mati. Settings -> Accelerator -> GPU T4, lalu Run All lagi.")


## Seperti apa isi container

Diambil apa adanya dari halaman *Code submission format* dan `README.md` runtime repo — bukan asumsi:

```
/code_execution/          <- working directory
├── data/                 <- read-only
│   ├── clips/*.mp3
│   └── test_metadata.csv     audio_filename, file_duration_seconds, language
├── src/                  <- isi submission.zip di-unzip ke sini
└── submission/           <- tulis submission.csv DI SINI
```

> **Koreksi atas brief 08 kita.** Brief menulis output ke `/code_execution/src/`. Itu salah — `src/` tempat kode, `submission/` tempat hasil. Brief juga belum tahu ada `test_metadata.csv`.

Entrypoint menjalankan `uv run src/main.py` dengan cwd `/code_execution`. Sel 7 menirunya persis, sebagai subprocess, supaya `main.py` tidak bisa diam-diam bergantung pada apa pun yang sudah diimpor notebook ini.

**Jalur decode `main.py` adalah salinan kode evaluasi NB 06 (sel 1 dan 7 di sana), konstanta demi konstanta:** batas klip panjang 29,8 s, jendela 28 s dengan overlap 2 s, dedup sambungan sampai 8 kata, loader librosa, batch 8 dalam urutan manifest, `language="jw"`. Kesamaannya sudah diuji di laptop terhadap kode NB 06 itu sendiri, dengan transcriber palsu: batch dan teksnya identik.

**Yang berbeda antar arm hanya dua hal:** isi folder `model/` di ZIP, dan satu baris `NUM_BEAMS = …` di `main.py`. Dengan `NUM_BEAMS = 1` kodenya men-decode persis seperti submission pertama. Kontrol C membuktikannya ulang di run ini.


In [ ]:
# ============================================================
# CELL 2 - DISCOVERY
# ============================================================
import os, json, shutil, time, csv, re, hashlib, zipfile, collections, gc, difflib
from pathlib import Path
import numpy as np, pandas as pd

def head(t):
    print("\n" + "=" * 74); print(t); print("=" * 74)

head("DISCOVERY")
WORKING = Path("/kaggle/working")
TEMP    = Path("/kaggle/temp"); TEMP.mkdir(parents=True, exist_ok=True)
RUN_ID  = NOTEBOOK_ID + (("-" + RUN_NOTE) if RUN_NOTE else "")
OUT_DIR = WORKING / "findings" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Mount discovery is content-based, not path-based: this account mounts
# owner-scoped and the depth has changed before. Same rule as NB 06.
INPUT = Path("/kaggle/input")
_hits = sorted(INPUT.glob("**/indonesian_dev/metadata.tsv")) or \
        sorted(p for p in INPUT.glob("**/metadata.tsv") if (p.parent / "clips").is_dir())
if not _hits:
    raise FileNotFoundError("dev set tidak ditemukan - attach dataset indonesian-dev")
DEV_META  = _hits[0]
DEV_CLIPS = DEV_META.parent / "clips"
print("dev metadata :", DEV_META)
print("dev clips    :", DEV_CLIPS, "|", len(list(DEV_CLIPS.glob('*.mp3'))), "mp3")

DEV = pd.read_csv(DEV_META, sep="\t", dtype=str, keep_default_na=False)
print("dev rows     :", len(DEV), "| language:", DEV["language"].value_counts().to_dict())
assert len(DEV) == 372, "dev resmi seharusnya 372 klip, dapat %d" % len(DEV)
print("arm          :", " | ".join("%s = %s, beam %d%s" % (k, v["ckpt"], v["num_beams"],
                                    "" if v["ship"] else " (kontrol, tanpa ZIP)")
                                    for k, v in ARMS.items()))

from kaggle_secrets import UserSecretsClient
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN     : ada")
except Exception as e:
    HF_TOKEN = None
    print("HF_TOKEN     : TIDAK ADA ->", e)
    print("!!! Secret Kaggle bersifat PER-NOTEBOOK. Add-ons -> Secrets -> HF_TOKEN.")
    print("!!! Tanpa ini bobot (repo private) tidak bisa diunduh dan notebook berhenti di sel 3.")

FINDINGS = {}
def rec(section, **kw):
    FINDINGS[section] = {**FINDINGS.get(section, {}), **kw}

def dump_handoff():
    # Blok yang disalin ke chat. Dicetak di sel 11, dan juga di sel 7 kalau
    # main.py gagal - Run All berhenti di sana, sel 11 tidak akan sempat jalan.
    blob = json.dumps(FINDINGS, ensure_ascii=False, separators=(",", ":"), sort_keys=True, default=str)
    print("\n===== BEGIN CLAUDE-HANDOFF-JSON =====")
    print(blob)
    print("===== END CLAUDE-HANDOFF-JSON =====")
    print("size: %.1f KB" % (len(blob) / 1024))

def sha16(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for blk in iter(lambda: fh.read(1 << 22), b""): h.update(blk)
    return h.hexdigest()[:16]

rec("env", transformers=transformers.__version__, torch=torch.__version__,
    numpy=numpy.__version__, python=platform.python_version(), pinned=bool(PIN_TRANSFORMERS),
    gpu=torch.cuda.get_device_name(0), n_dev=len(DEV), n_dev_used=(N_DEV or len(DEV)),
    ship_fp16=bool(SHIP_FP16),
    arms={k: dict(ckpt=v["ckpt"], num_beams=v["num_beams"], ship=v["ship"], parity=v["parity"])
          for k, v in ARMS.items()})


In [ ]:
# ============================================================
# CELL 3 - BOBOT -> /kaggle/temp/ckpt_<nama>/model, satu kali per checkpoint
# ============================================================
head("BOBOT")
from huggingface_hub import snapshot_download
from safetensors import safe_open
from safetensors.torch import load_file, save_file

def _j(model_dir, name):
    p = model_dir / name
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}

def fetch_checkpoint(key):
    c = CHECKPOINTS[key]
    model_dir = TEMP / ("ckpt_" + key) / "model"
    if model_dir.parent.exists(): shutil.rmtree(model_dir.parent)
    model_dir.mkdir(parents=True)
    t0 = time.time()
    snapshot_download(repo_id=c["repo"], revision=c["rev"], token=HF_TOKEN, local_dir=str(model_dir))
    dl_s = time.time() - t0
    # Snapshot membawa pembukuan repo (.cache/, .gitattributes) yang tidak ikut dikirim,
    # dan training_args.bin: pickle dari Trainer yang tidak pernah dibaca saat inference.
    if (model_dir / ".cache").exists(): shutil.rmtree(model_dir / ".cache")
    for p in list(model_dir.iterdir()):
        if p.is_file() and (p.name.startswith(".git") or p.name == "training_args.bin"):
            p.unlink()
    print("repo     :", c["repo"], "@", c["rev"][:12], "| unduh %.1f s" % dl_s)

    # Notebook pelatihnya mencatat sidik jari bobot yang dipush. Kalau berbeda, yang
    # kita kemas bukan model yang menghasilkan angka di REFS - berhenti, jangan dikemas.
    st = model_dir / "model.safetensors"
    got = sha16(st)
    fp32_gb = st.stat().st_size / 1e9
    print("model.safetensors sha256[:16] =", got,
          ("-> COCOK dengan %s" % REFS[key]["src"]) if got == c["sha16"] else "-> !!! TIDAK COCOK")
    assert got == c["sha16"], "%s: bobot yang diunduh bukan bobot yang diukur %s" % (key, REFS[key]["src"])

    fp16_exact, fp16_sha = None, None
    if SHIP_FP16:
        # Cast yang persis sama dengan yang terjadi saat main.py (dan NB 04/06) memuat
        # model ke fp16 - round-to-nearest, tensor per tensor - hanya dipindah ke
        # waktu packaging. File konfigurasi tidak disentuh.
        with safe_open(str(st), "pt") as f:
            st_meta = f.metadata() or {}
        sd32 = load_file(str(st))
        sd16 = {k: (v.half() if v.is_floating_point() else v) for k, v in sd32.items()}
        tmp = model_dir / "model.fp16.tmp"
        save_file(sd16, str(tmp), metadata={**st_meta, "format": "pt"})
        back = load_file(str(tmp))
        fp16_exact = set(back) == set(sd32) and all(torch.equal(back[k], sd16[k]) for k in sd32)
        dtypes = dict(collections.Counter(str(v.dtype).replace("torch.", "") for v in back.values()))
        n_tensors = len(back)
        del sd32, sd16, back
        gc.collect()
        tmp.replace(st)
        fp16_sha = sha16(st)
        print("fp16     : %.2f GB -> %.2f GB | %d tensor %s | identik dengan cast fp32->fp16: %s"
              % (fp32_gb, st.stat().st_size / 1e9, n_tensors, dtypes, fp16_exact))
        assert fp16_exact, "konversi fp16 tidak identik dengan cast yang dilakukan main.py"

    files = sorted(p for p in model_dir.rglob("*") if p.is_file())
    size = sum(p.stat().st_size for p in files)
    print("\nisi model/ : %d file, %.2f GB" % (len(files), size / 1e9))
    for p in files: print("   %-28s %8.1f MB" % (p.name, p.stat().st_size / 1e6))

    # Kedua checkpoint disimpan oleh transformers 5; container memuatnya dengan 4.x.
    # Yang dicatat di sini persis hal-hal yang main.py baca sendiri dari file-file ini,
    # supaya kalau load di sel 7 gagal, sebabnya sudah kelihatan tanpa run ulang.
    proc_cfg, tok_cfg = _j(model_dir, "processor_config.json"), _j(model_dir, "tokenizer_config.json")
    gen_cfg, mdl_cfg = _j(model_dir, "generation_config.json"), _j(model_dir, "config.json")
    if (model_dir / "preprocessor_config.json").exists():
        fe_home, fe_cfg = "preprocessor_config.json", _j(model_dir, "preprocessor_config.json")
    elif "feature_size" in proc_cfg:
        fe_home, fe_cfg = "processor_config.json", proc_cfg
    else:
        fe_home, fe_cfg = next((("processor_config.json:" + k, v) for k, v in proc_cfg.items()
                                if isinstance(v, dict) and "feature_size" in v), (None, {}))
    fmt = dict(
        files=[p.name for p in files],
        feature_extractor_in=fe_home, feature_size=fe_cfg.get("feature_size"),
        model_mel_bins=mdl_cfg.get("num_mel_bins"),
        processor_config_keys=sorted(proc_cfg)[:20],
        tokenizer_class=tok_cfg.get("tokenizer_class"),
        clean_up_tokenization_spaces=tok_cfg.get("clean_up_tokenization_spaces"),
        tokenizer_special_keys=sorted(k for k in tok_cfg if "special" in k),
        gen_language=gen_cfg.get("language"), gen_task=gen_cfg.get("task"),
        gen_has_jw=("<|jw|>" in (gen_cfg.get("lang_to_id") or {})),
        gen_no_repeat_ngram=gen_cfg.get("no_repeat_ngram_size"), gen_max_length=gen_cfg.get("max_length"),
        gen_forced_decoder_ids=gen_cfg.get("forced_decoder_ids"),
        saved_by_transformers=mdl_cfg.get("transformers_version"),
        config_dtype=mdl_cfg.get("dtype", mdl_cfg.get("torch_dtype")))
    print("\nformat checkpoint (yang dibaca main.py):")
    for k, v in fmt.items():
        if k != "files": print("   %-30s %s" % (k, v))
    assert fe_home is not None, "tidak ada setelan feature extractor di checkpoint %s" % key
    assert fmt["gen_has_jw"], "generation_config.json %s tidak punya <|jw|> - language='jw' tidak akan jalan" % key
    assert fmt["feature_size"] == fmt["model_mel_bins"], "mel bins feature extractor != model (%s)" % key

    rec("checkpoint." + key, repo=c["repo"], revision=c["rev"], safetensors_sha16=got,
        expected_sha16=c["sha16"], sha_ok=(got == c["sha16"]), fp32_gb=round(fp32_gb, 3),
        fp16=bool(SHIP_FP16), fp16_exact=fp16_exact, fp16_sha16=fp16_sha,
        n_files=len(files), size_gb=round(size / 1e9, 3), download_s=round(dl_s, 1), **fmt)
    return model_dir

MODELS = {}
for key in dict.fromkeys(a["ckpt"] for a in ARMS.values()):
    print("\n--- checkpoint %s (arm %s) ---" % (key, ", ".join(k for k, a in ARMS.items() if a["ckpt"] == key)))
    MODELS[key] = fetch_checkpoint(key)


In [ ]:
# ============================================================
# CELL 4 - TULIS main.py DAN postprocess.py, SATU main.py PER ARM
# ============================================================
head("SUBMISSION SOURCE")
SRC_MAIN = r'''# main.py - Lost in Transcription, Indonesian-Javanese track.
#
# Runs inside the organiser's container. Working directory is /code_execution:
#   data/clips/*.mp3        read-only test audio
#   data/test_metadata.csv  audio_filename, file_duration_seconds, language
#   src/                    this file, postprocess.py and ./model
#   submission/             where submission.csv must be written
#
# Constraints this file is written against, all taken from the competition's
# own pages rather than assumed:
#   - no network of any kind, so weights ship in ./model and offline flags are
#     set BEFORE transformers is imported (it reads them at import time)
#   - 2 h for the full set, 1 min for the 5-clip smoke test: the smoke budget is
#     import + weight load, not decode, so nothing heavy happens at module scope
#   - logs are capped at 500 lines x 300 chars AND logging anything about the
#     test data is grounds for disqualification, so only aggregate counts and
#     timings are printed - never a filename, a duration, or a transcript
#   - each clip is processed independently (competition rule 4): no state, no
#     sorting by content, no cross-clip normalisation
#
# The decode path is notebook 06's evaluation code (its cells 1 and 7), constant
# for constant: the dev WER this model was chosen on only carries over to the
# test set if the submission decodes exactly the same way.
import os

# Must precede the transformers/HF imports below.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Library warnings go to stderr, which counts against the same 500-line cap.
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

import collections
import csv
import json
import math
import sys
import time
import traceback
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

_T_IMPORT_START = time.time()
import numpy as np
import torch
from transformers import (GenerationConfig, WhisperFeatureExtractor,
                          WhisperForConditionalGeneration, WhisperTokenizerFast)

try:
    import librosa
except Exception:
    librosa = None
_T_IMPORT = time.time() - _T_IMPORT_START

HERE = Path(__file__).resolve().parent
WORK = Path.cwd()
DATA = WORK / "data"
CLIPS = DATA / "clips"
OUT_DIR = WORK / "submission"
MODEL_DIR = HERE / "model"

# Notebook 06, cell 1 (knobs) and cell 7 (decode helpers).
SR = 16000
LANG = "jw"                 # Whisper's Javanese code is "jw", not ISO "jv"
GEN_MAX_LENGTH = 225
NO_REPEAT_NGRAM = 4
BATCH = 8                   # EVAL_BATCH
LONG_S = 29.8               # clips longer than this take the chunked path
CHUNK_S = 28.0              # LONG_CHUNK_S
OVERLAP_S = 2.0             # LONG_OVERLAP_S
MIN_WINDOW_S = 0.3          # a trailing window this short is dropped
MAX_OVERLAP_WORDS = 8       # seam de-duplication, longest match first

# Beam width. 1 = greedy, the setting notebook 06 measured and every constant
# above was chosen under. The packaging notebook rewrites this one line per
# experiment arm and refuses an arm whose main.py differs anywhere else.
NUM_BEAMS = 1

GEN_KWARGS = dict(task="transcribe", language=LANG, num_beams=NUM_BEAMS, do_sample=False,
                  no_repeat_ngram_size=NO_REPEAT_NGRAM, max_length=GEN_MAX_LENGTH,
                  return_timestamps=False)

# A few unreadable clips cost a few points. A run where many clips fail means
# something systemic broke - a library mismatch, a missing file - and would
# hand the platform a scored file of blanks that burns a submission. Past this
# rate no submission.csv is written and the run exits non-zero; failed runs do
# not count against the submission limit.
MAX_FAIL_RATE = 0.02
MAX_FAIL_FLOOR = 2
FAIL_FAST_AFTER = 16

# Set only by our own test notebook. Never set on the platform, where a
# traceback could say something about a test file.
DEBUG = os.environ.get("LIT_DEBUG") == "1"


def log(msg):
    # Aggregate facts only. Never clip identities or transcript content.
    print(msg, flush=True)


def load_audio(path):
    # Notebook 06's load_official_audio: librosa first, soundfile + polyphase
    # resampling as the fallback. The same loader and the same resampler mean
    # the model sees the samples it was measured on.
    if librosa is not None:
        try:
            y, _ = librosa.load(str(path), sr=SR, mono=True)
            return y.astype(np.float32)
        except Exception:
            pass
    import soundfile as sf
    y, sr0 = sf.read(str(path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr0 != SR:
        from scipy.signal import resample_poly
        g = math.gcd(int(sr0), SR)
        y = resample_poly(y, SR // g, int(sr0) // g)
    return y.astype(np.float32)


def split_windows(y):
    # Fixed windows with an overlap, so a word straddling a boundary is seen
    # whole by at least one window. Whisper's encoder window is 30 s and
    # anything past it would be silently cut.
    step = int((CHUNK_S - OVERLAP_S) * SR)
    win = int(CHUNK_S * SR)
    parts = [y[i:i + win] for i in range(0, len(y), step)]
    return [p for p in parts if len(p) > int(MIN_WINDOW_S * SR)]


def join_overlapping(texts):
    # Consecutive windows usually repeat a few words at the seam. Drop the
    # longest repeated run, longest match first.
    joined = ""
    for t in texts:
        t = (t or "").strip()
        if not t:
            continue
        if not joined:
            joined = t
            continue
        a, b = joined.split(), t.split()
        k = 0
        for n in range(min(MAX_OVERLAP_WORDS, len(a), len(b)), 0, -1):
            if [w.lower() for w in a[-n:]] == [w.lower() for w in b[:n]]:
                k = n
                break
        joined = " ".join(a + b[k:])
    return joined.strip()


def _read_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


FE_KEYS = ("feature_size", "sampling_rate", "hop_length", "chunk_length", "n_fft",
           "padding_value", "dither")


def feature_extractor_config():
    # The checkpoint was saved by transformers 5, which writes a single
    # processor_config.json with the feature extractor nested inside it.
    # transformers 4 - the version the runtime pins - expects a separate
    # preprocessor_config.json. Reading the values by hand makes the file
    # layout of either major version irrelevant.
    pre = MODEL_DIR / "preprocessor_config.json"
    cfg = _read_json(pre) if pre.exists() else _read_json(MODEL_DIR / "processor_config.json")
    if "feature_size" in cfg:
        return cfg
    for v in cfg.values():
        if isinstance(v, dict) and "feature_size" in v:
            return v
    raise RuntimeError("no feature extractor settings found in ./model")


class Recogniser:
    def __init__(self):
        t0 = time.time()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.dtype = torch.float16 if self.device == "cuda" else torch.float32

        fe_cfg = feature_extractor_config()
        self.fe = WhisperFeatureExtractor(**{k: fe_cfg[k] for k in FE_KEYS if k in fe_cfg})
        # Built straight from tokenizer.json, which both major versions write in
        # the same format; tokenizer_config.json is the file whose format moved.
        # The only thing decoding needs from it is the clean-up flag.
        self.tok = WhisperTokenizerFast(tokenizer_file=str(MODEL_DIR / "tokenizer.json"))
        tok_cfg_path = MODEL_DIR / "tokenizer_config.json"
        tok_cfg = _read_json(tok_cfg_path) if tok_cfg_path.exists() else {}
        self.cleanup = bool(tok_cfg.get("clean_up_tokenization_spaces", False))

        model, info = WhisperForConditionalGeneration.from_pretrained(
            str(MODEL_DIR), local_files_only=True, dtype=self.dtype, output_loading_info=True)
        # With library warnings silenced, a checkpoint whose tensor names do not
        # match would load as random weights without a word. Refuse instead. Two
        # tensors may legitimately be absent from a file saved by another major
        # version: the output projection (tied to the token embedding) and the
        # encoder's fixed sinusoidal positions, which the library recomputes.
        derived = ("proj_out.weight", "encoder.embed_positions.weight")
        missing = info.get("missing_keys", [])
        self.n_derived = sum(1 for k in missing if k.endswith(derived))
        bad = len(missing) - self.n_derived + len(info.get("mismatched_keys", []))
        if bad:
            raise RuntimeError("checkpoint does not fit the model: %d tensors missing or mismatched" % bad)
        self.n_unexpected = len(info.get("unexpected_keys", []))
        # The explicit cast makes the dtype independent of which keyword this
        # transformers release honours (dtype vs torch_dtype).
        self.model = model.to(device=self.device, dtype=self.dtype).eval()
        self.model.config.use_cache = True
        if getattr(self.model.config, "forced_decoder_ids", None) is not None:
            self.model.config.forced_decoder_ids = None
        if self.fe.feature_size != self.model.config.num_mel_bins:
            raise RuntimeError("feature extractor has %d mel bins, model expects %d"
                               % (self.fe.feature_size, self.model.config.num_mel_bins))

        # generation_config.json carries Whisper's language table; without it
        # language="jw" cannot be honoured. Loaded explicitly so that a file this
        # release cannot parse stops the run instead of silently decoding with a
        # default config.
        gc = GenerationConfig.from_pretrained(str(MODEL_DIR), local_files_only=True)
        if "<|%s|>" % LANG not in (getattr(gc, "lang_to_id", None) or {}):
            raise RuntimeError("generation_config.json has no <|%s|> entry" % LANG)
        # Notebook 06's apply_gen_config: language and task pinned, the legacy
        # forced_decoder_ids path switched off so it cannot silently win.
        gc.language = LANG
        gc.task = "transcribe"
        gc.forced_decoder_ids = None
        gc.num_beams = NUM_BEAMS
        gc.do_sample = False
        gc.no_repeat_ngram_size = NO_REPEAT_NGRAM
        gc.max_length = GEN_MAX_LENGTH
        gc.use_cache = True
        self.model.generation_config = gc
        self.t_load = time.time() - t0

    @torch.no_grad()
    def transcribe(self, arrays):
        feats = self.fe(arrays, sampling_rate=SR, return_tensors="pt").input_features
        ids = self.model.generate(feats.to(self.device, dtype=self.dtype), **GEN_KWARGS)
        return [t.strip() for t in self.tok.batch_decode(
            ids, skip_special_tokens=True, clean_up_tokenization_spaces=self.cleanup)]


def read_names():
    meta_path = DATA / "test_metadata.csv"
    if meta_path.exists():
        with open(meta_path, newline="", encoding="utf-8") as fh:
            return [r["audio_filename"] for r in csv.DictReader(fh)]
    # The manifest is documented, but walking clips/ costs nothing and means
    # a missing or renamed manifest degrades instead of failing the run.
    return sorted(p.name for p in CLIPS.glob("*.mp3"))


def main():
    t_start = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    names = read_names()
    log("clips to transcribe: %d" % len(names))

    rec = Recogniser()
    log("import %.1fs | model load %.1fs | device %s | %s | tensors recomputed %d, unexpected %d"
        % (_T_IMPORT, rec.t_load, rec.device, str(rec.dtype).replace("torch.", ""),
           rec.n_derived, rec.n_unexpected))

    from postprocess import postprocess

    results = {}
    stats = collections.Counter()
    errors = collections.Counter()
    buf_names, buf_audio = [], []

    def failed(exc):
        stats["failed"] += 1
        errors[type(exc).__name__] += 1
        if DEBUG and errors[type(exc).__name__] == 1:
            traceback.print_exc()

    def run_short():
        # One generate() call per BATCH short clips in manifest order - the batch
        # composition notebook 06 measured. If the batch throws, each clip is
        # retried alone so one bad input cannot blank its neighbours.
        try:
            texts = rec.transcribe(buf_audio)
        except Exception:
            texts = []
            for a in buf_audio:
                try:
                    texts.append(rec.transcribe([a])[0])
                except Exception as e:
                    failed(e)
                    texts.append(None)
        for nm, tx in zip(buf_names, texts):
            results[nm] = "" if tx is None else postprocess(tx)
        stats["short"] += len(buf_names)
        buf_names.clear()
        buf_audio.clear()

    for name in names:
        try:
            y = load_audio(CLIPS / name)
        except Exception as e:
            failed(e)
            results[name] = ""
        else:
            if len(y) / SR > LONG_S:
                # Decoded on the spot, without flushing the short-clip buffer
                # first: notebook 06 did the same, so batches line up clip for clip.
                try:
                    parts = split_windows(y)
                    results[name] = (postprocess(join_overlapping(rec.transcribe(parts)))
                                     if parts else "")
                    stats["chunked"] += 1
                    stats["windows"] += len(parts)
                except Exception as e:
                    failed(e)
                    results[name] = ""
            else:
                buf_names.append(name)
                buf_audio.append(y)
                if len(buf_names) == BATCH:
                    run_short()
        if len(results) >= FAIL_FAST_AFTER and stats["failed"] == len(results):
            raise RuntimeError("the first %d clips all failed (%s) - stopping before a blank file"
                               " is scored" % (len(results), ", ".join(sorted(errors))))
    if buf_names:
        run_short()

    n = len(names)
    n_empty = sum(1 for nm in names if not results.get(nm, "").strip())
    log("short %d | chunked %d (%d windows) | failed %d | empty %d | %d clips in %.1fs"
        % (stats["short"], stats["chunked"], stats["windows"], stats["failed"], n_empty,
           n, time.time() - t_start))
    if errors:
        log("failure types: %s" % ", ".join("%s x%d" % kv for kv in sorted(errors.items())))
    if stats["failed"] > max(MAX_FAIL_FLOOR, int(MAX_FAIL_RATE * n)):
        log("too many failed clips - no submission.csv written, exiting non-zero")
        return 1

    # csv.writer, not string formatting: transcripts contain commas and quotes,
    # and the platform rejects a hand-assembled file.
    out = OUT_DIR / "submission.csv"
    with open(out, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["audio_filename", "transcript"])
        for nm in names:
            w.writerow([nm, results.get(nm, "")])
    log("wrote %d rows" % n)
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''
SRC_POST = r'''# postprocess.py - text fixes applied to every hypothesis before it is written.
#
# DELIBERATELY ALMOST EMPTY IN PHASE 1. The orthographic rules belong to
# notebook 07, and each one has to earn its place with a paired bootstrap on the
# official dev set before it goes in here. A rule that "looks obviously right"
# is how a submission quietly gets worse: the E4 error table shows the reference
# disagreeing with itself in both directions (gak->nggak 62 times, nggak->gak 33
# times), so a blanket rewrite fixes one column and breaks the other.
#
# What IS safe now is whitespace: it cannot change a token, only how tokens are
# separated, and the official scorer collapses runs of spaces anyway.
#
# Notebook 07 fills RULES below. Keep every entry measurable in isolation.
RULES = []


def postprocess(text):
    if not text:
        return ""
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    for pattern, replacement in RULES:
        text = pattern.sub(replacement, text)
    return " ".join(text.split()).strip()
'''
import ast
for fname, text in (("main.py", SRC_MAIN), ("postprocess.py", SRC_POST)):
    ast.parse(text)
    print("  %-16s %5d baris  sintaks OK" % (fname, text.count(chr(10))))
tree = ast.parse(SRC_MAIN)

# Jalur decode harus konstanta demi konstanta sama dengan NB 06 (sel 1 dan 7 di sana).
EXPECT_NB06 = dict(SR=16000, LANG="jw", GEN_MAX_LENGTH=225, NO_REPEAT_NGRAM=4, BATCH=8,
                   LONG_S=29.8, CHUNK_S=28.0, OVERLAP_S=2.0, MIN_WINDOW_S=0.3,
                   MAX_OVERLAP_WORDS=8, NUM_BEAMS=1)
consts = {t.id: ast.literal_eval(n.value) for n in tree.body if isinstance(n, ast.Assign)
          for t in n.targets if isinstance(t, ast.Name) and t.id in EXPECT_NB06}
const_diff = {k: (consts.get(k), v) for k, v in EXPECT_NB06.items() if consts.get(k) != v}
print("\nkonstanta decode vs NB 06 :", const_diff or "sama semua")

# Aturan kompetisi yang paling gampang dilanggar tanpa sadar: mencetak apa pun
# tentang data test. Di sumbernya: satu-satunya print() ada di dalam log(), dan
# tidak ada log() yang menyentuh variabel per-klip. (traceback hanya dicetak kalau
# LIT_DEBUG=1, yang tidak pernah di-set di platform.) Di keluarannya: sel 7
# mencari nama klip dev di stdout/stderr run sungguhan.
PER_CLIP = {"name", "nm", "y", "a", "tx", "texts", "parts", "results", "buf_names",
            "buf_audio", "path"}
n_print, bad_logs = 0, []
for node in ast.walk(tree):
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
        if node.func.id == "print":
            n_print += 1
        elif node.func.id == "log":
            used = {n.id for arg in node.args for n in ast.walk(arg) if isinstance(n, ast.Name)}
            if used & PER_CLIP:
                bad_logs.append((node.lineno, sorted(used & PER_CLIP)))
print("print() di main.py        : %d (harus 1, yaitu di dalam log())" % n_print)
print("log() dgn variabel klip   :", bad_logs or "tidak ada")
assert not const_diff, "main.py tidak lagi men-decode seperti NB 06"
assert n_print == 1 and not bad_logs, "main.py bisa mencetak sesuatu tentang data test"
rec("source", main_lines=SRC_MAIN.count("\n"), main_sha16=hashlib.sha256(SRC_MAIN.encode()).hexdigest()[:16],
    consts_match_nb06=not const_diff, n_print=n_print, n_bad_logs=len(bad_logs))

# Satu main.py per arm, diturunkan dari SRC_MAIN dengan mengganti SATU baris. Diff-nya
# diperiksa, bukan diasumsikan: baris lain yang ikut berubah berarti arm itu tidak lagi
# men-decode seperti yang terbukti 372/372 identik dengan NB 06.

# Per arm: postprocess.py = SRC_POST (dasar, persis) + opsional blok tabel rewrite DI BELAKANGNYA.
# Blok itu membungkus postprocess() dasar. Arm tanpa kunci "rewrite" memakai SRC_POST apa adanya,
# jadi perilakunya sama dengan semua submission sebelumnya.
REWRITE_BLOCK = r'''

# ---- rewrite table, arm {arm} (added by the packaging notebook) ----
# Learned from this checkpoint's own predictions on the 372 official dev clips: a hypothesis
# token is rewritten only if the same reference spelling replaced it at least twice, and more
# often than the token was right. Applied in the official scorer's token space: the text is
# normalised exactly as score.py does it (the normaliser is idempotent), then mapped per token.
import re as _re

_REWRITE = {table}


def _lsi(m):
    delimiter, first, second = m.group(1), m.group(2), m.group(3)
    if first.isupper() and not (second and second.isupper()): first = first.lower()
    return delimiter + first + second

def _normalize_text(text):
    BRACKETED = _re.compile(r"\[[^\]]+\]"); UNINTELLIGIBLE_PAREN = _re.compile(r"\(\?+\)")
    WORD_PAREN = _re.compile(r"\(([^()]*)\)"); PUNCTUATION_OTHER = _re.compile('[¿¡";:]+')
    COMMA = _re.compile(",+"); SENTENCE_INITIAL = _re.compile(r"(^\s*|[.!?—]\s*)([^\W\d_])([^\W\d_]?)")
    SENTENCE_END = _re.compile("[!?]+"); MULTISPACE = _re.compile("  +")
    text = text.replace("~", "")
    text = _re.sub(BRACKETED, " ", text); text = _re.sub(UNINTELLIGIBLE_PAREN, " ", text)
    text = _re.sub(WORD_PAREN, r"\1", text); text = text.replace("#x27;", "'")
    text = _re.sub(PUNCTUATION_OTHER, " ", text)
    text = _re.sub(SENTENCE_INITIAL, _lsi, text)
    text = text.replace("—", ", "); text = _re.sub(COMMA, " ", text)
    text = _re.sub(SENTENCE_END, " ", text)
    text = text.replace("...", "!ELLIPSIS!").replace(".", " ").replace("!ELLIPSIS!", "...")
    while " ... " in text: text = text.replace(" ... ", " ")
    return _re.sub(MULTISPACE, " ", text)


_postprocess_base = postprocess


def postprocess(text):
    text = _postprocess_base(text)
    if not text:
        return ""
    return " ".join(_REWRITE.get(w, w) for w in _normalize_text(text).split())
'''


def post_for(arm, a):
    if not a.get("rewrite"):
        return SRC_POST
    tbl = dict(sorted(REWRITE_TABLES[a["rewrite"]].items()))
    return SRC_POST + REWRITE_BLOCK.replace("{arm}", arm).replace("{table}", repr(tbl))

print()
MAIN_TEXT, SRC_DIRS = {}, {}
for arm, a in ARMS.items():
    nb = int(a["num_beams"])
    text, n_sub = re.subn(r"^NUM_BEAMS = 1$", "NUM_BEAMS = %d" % nb, SRC_MAIN, flags=re.M)
    assert n_sub == 1, "baris NUM_BEAMS di main.py tidak ditemukan tepat satu kali"
    changed = [l for l in difflib.ndiff(SRC_MAIN.splitlines(), text.splitlines()) if l[:2] in ("- ", "+ ")]
    expect = [] if nb == 1 else ["- NUM_BEAMS = 1", "+ NUM_BEAMS = %d" % nb]
    got_nb = [ast.literal_eval(n.value) for n in ast.parse(text).body if isinstance(n, ast.Assign)
              and any(isinstance(t, ast.Name) and t.id == "NUM_BEAMS" for t in n.targets)]
    ok = changed == expect and got_nb == [nb]
    d = TEMP / ("src_" + arm)
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True)
    (d / "main.py").write_text(text, encoding="utf-8")
    post = post_for(arm, a)
    ast.parse(post)
    _ns = {}
    exec(compile(post, "postprocess_%s.py" % arm, "exec"), _ns)
    assert _ns["postprocess"]("") == "" and _ns["postprocess"]("  a  b ") == "a b"
    (d / "postprocess.py").write_text(post, encoding="utf-8")
    MAIN_TEXT[arm], SRC_DIRS[arm] = text, d
    sha = hashlib.sha256(text.encode()).hexdigest()[:16]
    print("  arm %s: NUM_BEAMS=%d | beda dari dasar: %s | main.py sha16 %s"
          % (arm, nb, ("tidak ada" if not changed else " / ".join(changed)) if ok else "!!! %s" % changed, sha))
    n_rw = len(REWRITE_TABLES.get(a.get("rewrite"), {})) if a.get("rewrite") else 0
    print("         postprocess.py: %s | sha16 %s"
          % ("dasar + tabel rewrite %s (%d aturan)" % (a["rewrite"], n_rw) if n_rw else "dasar",
             hashlib.sha256(post.encode()).hexdigest()[:16]))
    rec("source." + arm, num_beams=nb, main_sha16=sha, only_beams_differs=ok, diff=changed,
        post_sha16=hashlib.sha256(post.encode()).hexdigest()[:16], rewrite=a.get("rewrite"),
        n_rewrite=n_rw, post_base_prefix=post.startswith(SRC_POST))
    assert ok, "main.py arm %s berbeda dari dasar di luar baris NUM_BEAMS" % arm


In [ ]:
# ============================================================
# CELL 5 - PACK, satu ZIP per arm
# ============================================================
head("PACK")
# Arm yang dikirim -> /kaggle/working: hanya folder itu yang bisa diunduh dari panel
# Output Kaggle. Kontrol C -> /kaggle/temp, dihapus begitu run dev-nya selesai.
NEED = ["main.py", "postprocess.py", "model/config.json", "model/model.safetensors",
        "model/generation_config.json", "model/tokenizer.json"]

def pack(arm):
    a = ARMS[arm]
    zp = (WORKING if a["ship"] else TEMP) / a["zip"]
    if zp.exists(): zp.unlink()
    model_dir = MODELS[a["ckpt"]]
    entries = [(SRC_DIRS[arm] / "main.py", "main.py"), (SRC_DIRS[arm] / "postprocess.py", "postprocess.py")]
    entries += [(p, "model/" + p.relative_to(model_dir).as_posix())
                for p in sorted(model_dir.rglob("*")) if p.is_file()]
    t0 = time.time()
    with zipfile.ZipFile(zp, "w") as z:
        for p, arc in entries:
            # Bobot hampir tidak bisa dikompres; menyimpannya apa adanya menghemat
            # semenit CPU. Sisanya kecil dan di-deflate.
            ct = zipfile.ZIP_STORED if p.suffix == ".safetensors" else zipfile.ZIP_DEFLATED
            z.write(p, arc, compress_type=ct)      # main.py di ROOT, tanpa folder pembungkus
    with zipfile.ZipFile(zp) as z:
        names = z.namelist()
        bad = z.testzip()                       # baca ulang semua isi, cek CRC
        main_ok = "main.py" in names and z.read("main.py").decode("utf-8") == MAIN_TEXT[arm]
    sha, gb = sha16(zp), zp.stat().st_size / 1e9
    missing = [n for n in NEED if n not in names]
    print("arm %s -> %s" % (arm, zp))
    print("   %.2f GB, %d entri, %.0f s | sha256[:16] %s" % (gb, len(names), time.time() - t0, sha))
    print("   wajib ada: %s | CRC: %s | main.py di ZIP = main.py arm %s: %s"
          % ("lengkap" if not missing else "!!! HILANG %s" % missing,
             "OK" if bad is None else "!!! rusak: %s" % bad, arm, main_ok))
    assert not missing and bad is None and main_ok
    assert not any("/" in n and not n.startswith("model/") for n in names), "ada folder pembungkus"
    rec("zip." + arm, path=str(zp), shipped=bool(a["ship"]), size_gb=round(gb, 3), sha16=sha,
        n_entries=len(names), main_at_root=("main.py" in names), crc_ok=(bad is None),
        main_matches=main_ok, entries=names)
    return zp

ZIPS = {arm: pack(arm) for arm in ARMS}


In [ ]:
# ============================================================
# CELL 6 - /code_execution TIRUAN
# ============================================================
head("CONTAINER TIRUAN")
import soundfile as sf, librosa

_DUR = {}
def dur_s(path):
    # file_duration_seconds di manifest resmi berupa int. Di-cache per file sumber,
    # karena klip dev yang sama disalin ke container setiap arm.
    key = str(path)
    if key not in _DUR:
        try:
            info = sf.info(key); _DUR[key] = int(round(info.frames / info.samplerate))
        except Exception:
            try: _DUR[key] = int(round(librosa.get_duration(path=key)))
            except Exception: _DUR[key] = 0
    return _DUR[key]

LANG_OF = dict(zip(DEV["audio_filename"], DEV["language"]))

def build_container(root, clip_names, clip_src, zip_path, language=None):
    # data/clips + data/test_metadata.csv (kolom persis yang didokumentasikan)
    # + src/ (isi ZIP arm itu, yang BENAR-BENAR akan diunggah) + submission/ kosong.
    if root.exists(): shutil.rmtree(root)
    (root / "data" / "clips").mkdir(parents=True); (root / "submission").mkdir(parents=True)
    rows = []
    for n in clip_names:
        shutil.copy(clip_src / n, root / "data" / "clips" / n)
        rows.append(dict(audio_filename=n, file_duration_seconds=dur_s(clip_src / n),
                         language=language or LANG_OF.get(n, "javind")))
    meta = pd.DataFrame(rows, columns=["audio_filename", "file_duration_seconds", "language"])
    meta.to_csv(root / "data" / "test_metadata.csv", index=False)
    with zipfile.ZipFile(zip_path) as z: z.extractall(root / "src")
    return meta

def drop_src(root):
    # Bobot hasil unzip (1,6 GB per container) tidak dibutuhkan lagi setelah main.py
    # selesai. data/ dan submission/ tetap, untuk sel-sel berikutnya.
    shutil.rmtree(root / "src", ignore_errors=True)

CE_ROOT = TEMP / "code_execution"
if CE_ROOT.exists(): shutil.rmtree(CE_ROOT)
sub = DEV if not N_DEV else DEV.head(N_DEV)
d = pd.Series([dur_s(DEV_CLIPS / n) for n in sub["audio_filename"]])
print("klip per arm : %d" % len(sub))
print("durasi       : mean %.1fs, median %.1fs, max %ds, >30s: %d klip (%.1f%%)"
      % (d.mean(), d.median(), d.max(), (d > 30).sum(), 100 * (d > 30).mean()))
print("container    : %s/<arm>, dibangun di sel 7 satu per satu" % CE_ROOT)


In [ ]:
# ============================================================
# CELL 7 - JALANKAN main.py SEPERTI ENTRYPOINT, SATU ARM SATU CONTAINER
# ============================================================
head("RUN")
# Entrypoint resmi: cwd=/code_execution, lalu `uv run src/main.py`. Dijalankan
# sebagai subprocess supaya state notebook tidak ikut membantu - kalau main.py
# diam-diam bergantung pada sesuatu yang sudah diimpor di sini, di sinilah gagal.
# Env offline meniru container tanpa jaringan; Internet Kaggle tetap menyala.
ENV = {k: v for k, v in os.environ.items() if k != "LIT_DEBUG"}
ENV.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1")

def run_main(root, timeout, debug=False):
    env = dict(ENV, **({"LIT_DEBUG": "1"} if debug else {}))
    t0 = time.time()
    p = subprocess.run([sys.executable, "src/main.py"], cwd=str(root), env=env,
                       capture_output=True, text=True, timeout=timeout)
    return p, time.time() - t0

def load_times(stdout):
    m = re.search(r"import ([\d.]+)s \| model load ([\d.]+)s", stdout)
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)

def decode_paths(stdout):
    m = re.search(r"short (\d+) \| chunked (\d+) \((\d+) windows\) \| failed (\d+) \| empty (\d+)", stdout)
    return dict(zip(("short", "chunked", "windows", "failed", "empty"), map(int, m.groups()))) if m else {}

RUNS, FAILED = {}, []
n_clips = len(sub)
for arm, a in ARMS.items():
    print("\n--- arm %s: %s ---" % (arm, a["what"]))
    root = CE_ROOT / arm
    build_container(root, list(sub["audio_filename"]), DEV_CLIPS, ZIPS[arm])
    proc, wall = run_main(root, 7200)
    drop_src(root)
    if not a["ship"]:
        ZIPS[arm].unlink(missing_ok=True)      # kontrol: ZIP-nya tidak pernah diunggah
    log_all = proc.stdout + proc.stderr
    print(proc.stdout[-1500:])
    if proc.returncode != 0:
        print("STDERR:\n", proc.stderr[-4000:])
        FAILED.append(arm)
    imp_s, load_s = load_times(proc.stdout)
    paths = decode_paths(proc.stdout)
    decode_s = wall - (imp_s or 0) - (load_s or 0)
    proj_min = (2118 * decode_s / max(n_clips, 1) + (imp_s or 0) + (load_s or 0)) / 60
    log_lines = log_all.splitlines()
    n_long_lines = sum(1 for l in log_lines if len(l) > 300)
    # Aturan: log tidak boleh memuat apa pun tentang data test. Diuji di keluaran
    # sungguhan: tidak boleh ada satu pun nama klip dev di stdout/stderr.
    leak_names = sum(1 for f in sub["audio_filename"] if f.rsplit(".", 1)[0] in log_all)

    print("exit code        :", proc.returncode)
    print("wall clock       : %.1f s untuk %d klip (import %s s, load %s s)" % (wall, n_clips, imp_s, load_s))
    print("jalur decode     :", paths)
    print("proyeksi 2.118 klip test di GPU ini: %.1f menit (limit 120; A100 container lebih cepat)" % proj_min)
    print("baris log        : %d stdout+stderr (limit 500), %d baris > 300 karakter" % (len(log_lines), n_long_lines))
    print("nama klip di log : %d" % leak_names)
    (OUT_DIR / ("run-log-main-%s.txt" % arm)).write_text(log_all[-20000:], encoding="utf-8")
    RUNS[arm] = dict(root=root, log=log_all, paths=paths)
    rec("run." + arm, exit_code=proc.returncode, wall_s=round(wall, 1), import_s=imp_s, load_s=load_s,
        s_per_clip=round(decode_s / max(n_clips, 1), 4), projected_test_min=round(proj_min, 1),
        paths=paths, n_log_lines=len(log_lines), n_long_lines=n_long_lines, leak_names=leak_names,
        stderr_tail=proc.stderr[-600:] if proc.returncode else "")
if FAILED:
    dump_handoff()
assert not FAILED, "main.py gagal di arm %s - kirim STDERR dan blok JSON di atas ke Claude" % FAILED


In [ ]:
# ============================================================
# CELL 8 - SKOR DENGAN NORMALIZER RESMI + BERPASANGAN VS KONTROL
# ============================================================
head("SKOR")
# Disalin apa adanya dari score.py di runtime repo penyelenggara. JANGAN diedit
# agar cocok dengan scorer.py kita - arah pembuktiannya justru sebaliknya.
def _lowercase_sentence_initial(m):
    delimiter, first, second = m.group(1), m.group(2), m.group(3)
    if first.isupper() and not (second and second.isupper()): first = first.lower()
    return delimiter + first + second

def normalize_text(text):
    BRACKETED = re.compile(r"\[[^\]]+\]"); UNINTELLIGIBLE_PAREN = re.compile(r"\(\?+\)")
    WORD_PAREN = re.compile(r"\(([^()]*)\)"); PUNCTUATION_OTHER = re.compile('[¿¡";:]+')
    COMMA = re.compile(",+"); SENTENCE_INITIAL = re.compile(r"(^\s*|[.!?—]\s*)([^\W\d_])([^\W\d_]?)")
    SENTENCE_END = re.compile("[!?]+"); MULTISPACE = re.compile("  +")
    text = text.replace("~", "")
    text = re.sub(BRACKETED, " ", text); text = re.sub(UNINTELLIGIBLE_PAREN, " ", text)
    text = re.sub(WORD_PAREN, r"\1", text); text = text.replace("#x27;", "'")
    text = re.sub(PUNCTUATION_OTHER, " ", text)
    text = re.sub(SENTENCE_INITIAL, _lowercase_sentence_initial, text)
    text = text.replace("—", ", "); text = re.sub(COMMA, " ", text)
    text = re.sub(SENTENCE_END, " ", text)
    text = text.replace("...", "!ELLIPSIS!").replace(".", " ").replace("!ELLIPSIS!", "...")
    while " ... " in text: text = text.replace(" ... ", " ")
    return re.sub(MULTISPACE, " ", text)

def _lev(r, h):
    prev = list(range(len(h) + 1))
    for i, a in enumerate(r, 1):
        cur = [i] + [0] * len(h)
        for j, b in enumerate(h, 1):
            cur[j] = prev[j - 1] if a == b else 1 + min(prev[j - 1], prev[j], cur[j - 1])
        prev = cur
    return prev[-1]

def corpus_wer(refs, hyps):
    e = n = 0
    for r, h in zip(refs, hyps):
        rt, ht = normalize_text(r).split(), normalize_text(h).split()
        e += _lev(rt, ht); n += len(rt)
    return e / max(n, 1), e, n

def clip_errors(refs, hyps):
    # Per klip: (error, token referensi). Jumlahnya sama dengan corpus_wer; versi per
    # klip dibutuhkan untuk bootstrap berpasangan di bawah.
    out = []
    for r, h in zip(refs, hyps):
        rt, ht = normalize_text(r).split(), normalize_text(h).split()
        out.append((_lev(rt, ht), len(rt)))
    return np.array(out, dtype=np.int64).reshape(-1, 2)

FULL = not N_DEV
REF = dict(zip(DEV["audio_filename"], DEV["transcript"]))
order = list(sub["audio_filename"])
# Subset seleksi checkpoint NB 06 direkonstruksi persis (terverifikasi di handoff NB 06).
SEL150 = set(DEV.sample(n=150, random_state=20260829)["audio_filename"])
CLEAN = np.array([x not in SEL150 for x in order])

ERRS = {}
for arm, a in ARMS.items():
    print("\n--- arm %s: %s ---" % (arm, a["what"]))
    csv_path = RUNS[arm]["root"] / "submission" / "submission.csv"
    PRED = pd.read_csv(csv_path, keep_default_na=False)
    print("baris submission.csv :", len(PRED), "| kolom:", list(PRED.columns))
    assert list(PRED.columns) == ["audio_filename", "transcript"], "arm %s: header tidak sesuai kontrak" % arm
    assert len(PRED) == len(order), "arm %s: jumlah baris != test_metadata.csv" % arm
    assert PRED["audio_filename"].is_unique, "arm %s: ada audio_filename ganda" % arm
    assert list(PRED["audio_filename"]) == order, "arm %s: daftar/urutan klip tidak sama" % arm
    HYP = dict(zip(PRED["audio_filename"], PRED["transcript"]))
    E = clip_errors([REF[x] for x in order], [HYP[x] for x in order])
    ERRS[arm] = E
    n_err, n_tok = int(E[:, 0].sum()), int(E[:, 1].sum())
    wer = n_err / max(n_tok, 1)
    wer_c = float(E[CLEAN, 0].sum() / max(E[CLEAN, 1].sum(), 1)) if CLEAN.any() else float("nan")
    n_empty = int((PRED["transcript"].str.strip() == "").sum())
    leak_text = sum(1 for h in PRED["transcript"] if len(h.split()) >= 4 and h in RUNS[arm]["log"])
    # Arm kontrol (parity) HARUS menyamai run tersimpan checkpoint-nya. Arm beam
    # dibandingkan dengan greedy checkpoint yang sama: selisihnya efek beam, bukan kegagalan.
    ref_key = a["parity"] or a["ckpt"]
    R0 = REFS[ref_key]
    paths_ok = {k: RUNS[arm]["paths"].get(k) for k in REF_PATHS} == REF_PATHS
    print("WER lewat main.py   : %.4f  (%d error / %d token)" % (wer, n_err, n_tok))
    print("%-19s : %.4f  (%s, greedy) -> selisih %+.4f, %+d error"
          % ("harus sama dengan" if a["parity"] else "dibandingkan dgn", R0["wer"], R0["src"],
             wer - R0["wer"], n_err - R0["n_err"]))
    print("222 klip bersih     : %.4f  vs %.4f -> selisih %+.4f" % (wer_c, R0["clean"], wer_c - R0["clean"]))
    print("jalur decode        :", RUNS[arm]["paths"], "| harus:", REF_PATHS)
    print("transkrip kosong    :", n_empty)
    print("transkrip di log    :", leak_text)
    shutil.copy(csv_path, OUT_DIR / ("submission_dev_%s.csv" % arm))
    rec("score." + arm, full=FULL, parity=a["parity"], ref=ref_key, wer_official_normalizer=round(wer, 6),
        n_errors=n_err, n_ref_tokens=n_tok, ref_wer=R0["wer"], delta=round(wer - R0["wer"], 6),
        delta_errors=n_err - R0["n_err"], wer_clean222=round(wer_c, 6), ref_clean222=R0["clean"],
        delta_clean=round(wer_c - R0["clean"], 6), paths_match=paths_ok, n_empty=n_empty,
        leak_text=leak_text)
if not FULL:
    print("\n!!! N_DEV=%d: subset. Angka di atas TIDAK bisa dibandingkan dengan run tersimpan." % N_DEV)

# Bootstrap berpasangan terhadap kontrol C (E4 greedy = submission pertama): klip sama,
# referensi sama, jadi yang di-resample adalah klipnya. Ingat ini dev - 4 penutur,
# 2 percakapan - dan submission pertama sudah menunjukkan dev proxy yang lemah untuk
# test. Gunanya memisahkan efek nyata dari noise DI DEV, bukan meramal skor test.
def paired(ex, ey, n_boot=N_BOOT, seed=20260924):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(ex), size=(n_boot, len(ex)))
    tok = ex[:, 1][idx].sum(1)
    dd = (ex[:, 0][idx].sum(1) - ey[:, 0][idx].sum(1)) / np.maximum(tok, 1)
    lo, hi = np.percentile(dd, [2.5, 97.5])
    point = (ex[:, 0].sum() - ey[:, 0].sum()) / max(ex[:, 1].sum(), 1)
    return dict(delta=round(float(point), 4), ci95=[round(float(lo), 4), round(float(hi), 4)],
                p_better=round(float((dd < 0).mean()), 3))

PAIRED = {}
if "C" in ERRS and FULL:
    print("\nberpasangan vs C (E4 greedy), %d resample. Negatif = lebih baik dari submission pertama:" % N_BOOT)
    for arm in ERRS:
        if arm == "C": continue
        PAIRED[arm] = dict(full372=paired(ERRS[arm], ERRS["C"]),
                           clean222=paired(ERRS[arm][CLEAN], ERRS["C"][CLEAN]))
        for k, v in PAIRED[arm].items():
            print("   %s - C  %-9s %+.4f  CI95 [%+.4f, %+.4f]  p(lebih baik) %.3f"
                  % (arm, k, v["delta"], v["ci95"][0], v["ci95"][1], v["p_better"]))
rec("paired_vs_C", n_boot=N_BOOT, **PAIRED)

# Efek beam pada checkpoint yang sama: arm beam vs arm greedy dengan ckpt yang sama
# (selain C). Untuk E4 angka ini pindah dari dev ke test hampir persis (-0,0050 / -0,0055).
PAIRED_BEAM = {}
if FULL:
    for arm, a in ARMS.items():
        g = next((k for k, b in ARMS.items()
                  if k != arm and b["ckpt"] == a["ckpt"] and b["num_beams"] == 1), None)
        if a["num_beams"] > 1 and g is not None:
            PAIRED_BEAM[arm + "-" + g] = dict(full372=paired(ERRS[arm], ERRS[g]),
                                              clean222=paired(ERRS[arm][CLEAN], ERRS[g][CLEAN]))
    if PAIRED_BEAM:
        print("\nefek beam, checkpoint yang sama:")
    for k2, pv in PAIRED_BEAM.items():
        for k, v in pv.items():
            print("   %s  %-9s %+.4f  CI95 [%+.4f, %+.4f]  p(lebih baik) %.3f"
                  % (k2, k, v["delta"], v["ci95"][0], v["ci95"][1], v["p_better"]))
rec("paired_beam", n_boot=N_BOOT, **PAIRED_BEAM)

print("\nsudah diketahui (bukan cek): E4 beam5 dev %.4f -> test %.4f | E4 greedy dev %.4f -> test %.4f"
      % (KNOWN["B_e4beam5"]["dev"], KNOWN["B_e4beam5"]["test"],
         KNOWN["E4_greedy"]["dev"], KNOWN["E4_greedy"]["test"]))
print("D (E4-native beam5): dev %.4f -> test %.4f (id %d)" % (KNOWN["D_e4native_beam5"]["dev"],
      KNOWN["D_e4native_beam5"]["test"], KNOWN["D_e4native_beam5"]["id"]))
print("Checkpoint yang dilatih di dev TIDAK bisa dinilai di dev - hanya slot test yang bisa.")


In [ ]:
# ============================================================
# CELL 9 - GLADI BERSIH SMOKE TEST (limit 1 menit), per ZIP yang dikirim
# ============================================================
head("SMOKE TEST")
# Lima klip smoke test resmi (competition/smoke_test_submission_format.csv) -
# kelimanya klip dev. Yang mengikat di sini adalah import + load bobot, bukan
# decode: 5 klip men-decode dalam hitungan detik, dengan beam pun. Skor smoke di
# platform boleh diabaikan (kata penyelenggara); ini murni uji benar-jalan dan waktu load.
# Nama kelima klip TIDAK ditulis di sini: aturan kompetisi melarang mempublikasikan Data, dan
# nama klip termasuk di dalamnya. Attach file resmi smoke_test_submission_format.csv (halaman
# data kompetisi) sebagai input Kaggle; daftar klipnya dibaca dari sana.
_smoke_csv = sorted(Path("/kaggle/input").glob("**/smoke_test_submission_format.csv"))
if not _smoke_csv:
    raise RuntimeError("smoke_test_submission_format.csv tidak ter-attach (lihat komentar di atas).")
SMOKE = list(pd.read_csv(_smoke_csv[0])["audio_filename"])
assert len(SMOKE) == 5, "smoke test resmi berisi 5 klip, bukan %d" % len(SMOKE)

def evict_page_cache(root):
    # Kaggle baru saja menulis file-file ini, jadi semuanya masih di page cache
    # dan load terlihat lebih cepat daripada di container, yang membaca bobot dari
    # disk dingin. fsync + POSIX_FADV_DONTNEED membuang cache per file tanpa root.
    # Library Python-nya sendiri tetap hangat - bagian itu tidak bisa ditiru.
    n = 0
    for p in root.rglob("*"):
        if p.is_file():
            fd = os.open(str(p), os.O_RDONLY)
            try:
                os.fsync(fd); os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED); n += 1
            finally:
                os.close(fd)
    return n

for arm, a in ARMS.items():
    if not a["ship"]:
        continue
    print("\n--- arm %s: %s ---" % (arm, a["what"]))
    CS = TEMP / ("code_execution_smoke_" + arm)
    build_container(CS, SMOKE, DEV_CLIPS, ZIPS[arm])
    n_evicted = evict_page_cache(CS)
    ps, smoke_s = run_main(CS, 600)
    print(ps.stdout[-1000:])
    if ps.returncode:
        print("STDERR:\n", ps.stderr[-3000:])
    s_imp, s_load = load_times(ps.stdout)
    verdict = ("AMAN" if smoke_s <= SMOKE_MARGIN_S else
               "MUAT, TAPI MEPET" if smoke_s <= SMOKE_LIMIT_S else "!!! LEWAT")
    print("cache dibuang  : %d file" % n_evicted)
    print("total          : %.1f s (limit %d s) -> %s" % (smoke_s, SMOKE_LIMIT_S, verdict))
    print("  import %s s | load bobot %s s | sisanya start Python + decode 5 klip" % (s_imp, s_load))
    if smoke_s > SMOKE_MARGIN_S:
        print("!!! Jalan keluar yang tersedia di image: faster-whisper 1.2.1 / ctranslate2 4.8.2.")
        print("!!! Tapi itu mengubah numerik decode, jadi WER harus diukur ulang. Lapor ke Claude dulu.")

    R0 = REFS[a["parity"] or a["ckpt"]]
    SP_PATH = CS / "submission" / "submission.csv"
    sw = None
    if ps.returncode == 0 and SP_PATH.exists():
        SP = pd.read_csv(SP_PATH, keep_default_na=False)
        sh = dict(zip(SP.audio_filename, SP.transcript))
        sw, _, _ = corpus_wer([REF[x] for x in SMOKE], [sh.get(x, "") for x in SMOKE])
        print("WER 5 klip     : %.4f  (%s greedy: %.4f - sanity, sampel terlalu kecil untuk dipercaya)"
              % (sw, R0["src"], R0["smoke"]))
    rec("smoke." + arm, exit_code=ps.returncode, wall_s=round(smoke_s, 1), import_s=s_imp, load_s=s_load,
        fits_limit=bool(ps.returncode == 0 and smoke_s <= SMOKE_LIMIT_S),
        within_margin=bool(smoke_s <= SMOKE_MARGIN_S), cache_evicted=n_evicted,
        wer=(round(sw, 4) if sw is not None else None), ref_wer=round(R0["smoke"], 4))
    shutil.rmtree(CS, ignore_errors=True)


In [ ]:
# ============================================================
# CELL 10 - INPUT ANEH + RUN RUSAK TOTAL, per ZIP yang dikirim
# ============================================================
head("EDGE CASES")
if not DO_EDGE:
    print("dilewati (DO_EDGE=False)")
else:
    STAGE = TEMP / "edge_stage"
    if STAGE.exists(): shutil.rmtree(STAGE)
    STAGE.mkdir(parents=True)
    donor, _ = librosa.load(str(DEV_CLIPS / DEV["audio_filename"].iloc[0]), sr=48000, mono=True)
    fmts = {}
    def put(name, arr, sr=48000):
        # MP3 sungguhan kalau libsndfile di sini bisa menulisnya; kalau tidak, WAV
        # bernama .mp3 - loader membaca isinya, bukan ekstensinya.
        try:
            sf.write(str(STAGE / name), arr, sr, format="MP3"); fmts[name] = "MP3"
        except Exception:
            sf.write(str(STAGE / name), arr, sr, format="WAV"); fmts[name] = "WAV"
    tile = lambda s: np.tile(donor, int(np.ceil(s * 48000 / len(donor))))[:int(s * 48000)]
    put("edge_silence.mp3", np.zeros(5 * 48000, dtype="float32"))
    put("edge_veryshort.mp3", donor[:int(0.3 * 48000)])
    put("edge_35s.mp3", tile(35))
    put("edge_40s.mp3", tile(40))
    put("edge_noise.mp3", (np.random.RandomState(0).randn(6 * 48000) * 0.05).astype("float32"))
    (STAGE / "edge_corrupt.mp3").write_bytes(b"not audio at all")
    edge_names = sorted(p.name for p in STAGE.iterdir())

    # Run rusak total: lima 'klip' yang semuanya bukan audio. main.py harus berhenti
    # dengan exit code != 0 dan TIDAK menulis submission.csv. Kalau tidak, di platform
    # ini jadi file kosong yang dinilai dan memakan satu jatah submission.
    STAGE2 = TEMP / "allbad_stage"
    if STAGE2.exists(): shutil.rmtree(STAGE2)
    STAGE2.mkdir(parents=True)
    bad_names = ["bad_%d.mp3" % i for i in range(5)]
    for n in bad_names: (STAGE2 / n).write_bytes(b"junk")

    for arm, a in ARMS.items():
        if not a["ship"]:
            continue
        print("\n--- arm %s: %s ---" % (arm, a["what"]))
        CX = TEMP / ("code_execution_edge_" + arm)
        build_container(CX, edge_names, STAGE, ZIPS[arm], language="javind")
        # LIT_DEBUG=1 hanya di sini: klipnya buatan sendiri, jadi traceback aman dibaca.
        pe, _ = run_main(CX, 1800, debug=True)
        print(pe.stdout[-1500:])
        if pe.stderr.strip():
            print("STDERR (traceback pertama per jenis error):\n", pe.stderr[-2500:])
        EP_PATH = CX / "submission" / "submission.csv"
        EP = pd.read_csv(EP_PATH, keep_default_na=False) if EP_PATH.exists() else \
             pd.DataFrame(columns=["audio_filename", "transcript"])
        words = {r.audio_filename: len(str(r.transcript).split()) for r in EP.itertuples()}
        for k in edge_names:
            print("   %-22s %-4s %4s kata" % (k, fmts.get(k, "raw"), words.get(k, "-")))
        ep = decode_paths(pe.stdout)
        EDGE = dict(exit_code=pe.returncode, n_clips=len(edge_names), n_rows=len(EP),
                    chunked=ep.get("chunked"), windows=ep.get("windows"), failed=ep.get("failed"),
                    corrupt_words=words.get("edge_corrupt.mp3"), formats=fmts,
                    words={k.replace(".mp3", ""): v for k, v in words.items()})
        print("\nrun tidak jatuh      :", pe.returncode == 0)
        print("semua klip terwakili :", len(EP) == len(edge_names))
        print("35 s & 40 s di-chunk :", ep.get("chunked") == 2, "(%s jendela)" % ep.get("windows"))
        print("rusak -> '' + gagal  :", words.get("edge_corrupt.mp3") == 0 and ep.get("failed") == 1)
        rec("edge." + arm, **EDGE)
        shutil.rmtree(CX, ignore_errors=True)

        CZ = TEMP / ("code_execution_allbad_" + arm)
        build_container(CZ, bad_names, STAGE2, ZIPS[arm], language="javind")
        pz, _ = run_main(CZ, 600)
        print("\n" + pz.stdout[-600:])
        ALLBAD = dict(exit_code=pz.returncode, wrote_csv=(CZ / "submission" / "submission.csv").exists())
        print("run rusak total      : exit %d, submission.csv %s"
              % (pz.returncode, "ADA (!!!)" if ALLBAD["wrote_csv"] else "tidak ditulis"))
        rec("allbad." + arm, **ALLBAD)
        shutil.rmtree(CZ, ignore_errors=True)


In [ ]:
# ============================================================
# CELL 11 - SELF-VALIDATION + ARTEFAK + HANDOFF
# ============================================================
head("SELF-VALIDATION")
CHECKS = []
def check(arm, name, cond, detail=""):
    CHECKS.append(dict(arm=arm, check=name, passed=bool(cond), detail=str(detail)[:180]))
    print("  [%s] %-58s %s" % ("PASS" if cond else "FAIL", name, detail))

SR_ = FINDINGS["source"]
print("--- semua arm ---")
check("*", "transformers cocok dengan container (<5)", MAJOR < 5, transformers.__version__)
check("*", "konstanta decode main.py dasar = NB 06", SR_["consts_match_nb06"])
check("*", "main.py tidak mencetak apa pun tentang klip", SR_["n_print"] == 1 and SR_["n_bad_logs"] == 0)

for arm, a in ARMS.items():
    print("\n--- arm %s: %s ---" % (arm, a["what"]))
    CK, ZP, M = FINDINGS["checkpoint." + a["ckpt"]], FINDINGS["zip." + arm], FINDINGS["source." + arm]
    R, S = FINDINGS["run." + arm], FINDINGS["score." + arm]
    check(arm, "bobot = sidik jari %s" % REFS[a["ckpt"]]["src"], CK["sha_ok"], CK["safetensors_sha16"])
    if SHIP_FP16:
        check(arm, "fp16 identik dengan cast fp32->fp16 saat load", CK["fp16_exact"], "%.2f GB" % CK["size_gb"])
    check(arm, "main.py beda dari dasar hanya di NUM_BEAMS = %d" % a["num_beams"], M["only_beams_differs"],
          " / ".join(M["diff"]) or "tidak ada beda")
    check(arm, "postprocess.py = dasar%s" % (" + tabel rewrite %s di belakang" % M["rewrite"] if M["rewrite"] else ""),
          M["post_base_prefix"], "%d aturan, sha16 %s" % (M["n_rewrite"], M["post_sha16"]))
    check(arm, "main.py + model di root arsip, CRC OK", ZP["main_at_root"] and ZP["crc_ok"] and ZP["main_matches"],
          "%.2f GB, %d entri" % (ZP["size_gb"], ZP["n_entries"]))
    check(arm, "main.py jalan sampai selesai", R["exit_code"] == 0)
    check(arm, "tidak ada transkrip kosong", S["n_empty"] == 0, "%d kosong" % S["n_empty"])
    check(arm, "jalur decode = 328 pendek / 44 chunk / 88 jendela", S["full"] and S["paths_match"], R["paths"])
    R0 = REFS[S["ref"]]
    if a["parity"]:
        check(arm, "WER lewat main.py = %s (toleransi %.3f)" % (R0["src"], PARITY_TOL),
              S["full"] and abs(S["delta"]) <= PARITY_TOL,
              "%.4f vs %.4f, selisih %+.4f (%+d error)" % (S["wer_official_normalizer"], R0["wer"],
                                                          S["delta"], S["delta_errors"]))
        check(arm, "222 klip bersih = %s (toleransi %.3f)" % (R0["src"], PARITY_TOL),
              S["full"] and abs(S["delta_clean"]) <= PARITY_TOL, "%.4f vs %.4f" % (S["wer_clean222"], R0["clean"]))
    else:
        # Bukan syarat menang: beam boleh kalah. Yang ditangkap di sini beam search yang
        # rusak (urutan hasil tertukar, output terpotong), bukan beam yang kurang bagus.
        check(arm, "beam tidak rusak: WER <= %s greedy + %.2f" % (R0["src"], BEAM_GUARD),
              S["full"] and S["delta"] <= BEAM_GUARD,
              "%.4f vs %.4f, selisih %+.4f" % (S["wer_official_normalizer"], R0["wer"], S["delta"]))
    check(arm, "proyeksi test muat di limit 2 jam", R["projected_test_min"] < 120,
          "%.1f menit di GPU ini" % R["projected_test_min"])
    # Limit resminya 500 baris, tapi run dev hanya 372 klip sedangkan test 2.118: satu
    # peringatan per batch yang lolos di sini jadi ~265 baris di sana. Jadi ukurannya
    # bukan "< 500" melainkan "tidak bertambah dengan jumlah klip" - main.py mencetak 4.
    check(arm, "log pendek (<= 20 baris), tiap baris <= 300 karakter",
          R["n_log_lines"] <= 20 and R["n_long_lines"] == 0, "%d baris" % R["n_log_lines"])
    check(arm, "log tidak memuat nama klip / transkrip", R["leak_names"] == 0 and S["leak_text"] == 0,
          "%d nama, %d transkrip" % (R["leak_names"], S["leak_text"]))
    if a["ship"]:
        SM = FINDINGS["smoke." + arm]
        check(arm, "smoke test muat di limit 1 menit", SM["fits_limit"],
              "%.1f s (import %s, load %s)%s" % (SM["wall_s"], SM["import_s"], SM["load_s"],
                                                 "" if SM["within_margin"] else " - MEPET"))
        if DO_EDGE:
            E, A_ = FINDINGS["edge." + arm], FINDINGS["allbad." + arm]
            check(arm, "input aneh tidak menjatuhkan run", E["exit_code"] == 0 and E["n_rows"] == E["n_clips"],
                  "%d/%d baris" % (E["n_rows"], E["n_clips"]))
            check(arm, "klip 35 s dan 40 s lewat jalur chunk", E["chunked"] == 2, "%s jendela" % E["windows"])
            check(arm, "klip rusak jadi '' dan dihitung gagal", E["corrupt_words"] == 0 and E["failed"] == 1)
            check(arm, "run rusak total berhenti tanpa submission.csv",
                  A_["exit_code"] != 0 and not A_["wrote_csv"], "exit %d" % A_["exit_code"])

# Siap = semua check arm itu + semua check umum PASS + SEMUA kontrol (arm tanpa ZIP)
# PASS. C membuktikan main.py masih submission pertama; N membuktikan checkpoint baru
# lewat main.py menyamai NB 06. Kalau satu saja gagal, tidak ada ZIP yang boleh dikirim.
own = {arm: all(c["passed"] for c in CHECKS if c["arm"] in ("*", arm)) for arm in ARMS}
CONTROLS_OK = all(own[k] for k, b in ARMS.items() if not b["ship"])
READY = {arm: own[arm] and CONTROLS_OK for arm in ARMS}
n_pass = sum(1 for c in CHECKS if c["passed"])
rec("validation", checks=CHECKS, n_passed=n_pass, n_total=len(CHECKS), ready=READY)

# Artefak kecil -> findings/08-packaging-ab (dicerminkan ke repo). ZIP tidak ikut di
# sini; masing-masing berdiri sendiri di /kaggle/working.
for arm in ARMS:
    shutil.copy(SRC_DIRS[arm] / "main.py", OUT_DIR / ("main_%s.py" % arm))
for arm in ARMS:
    shutil.copy(SRC_DIRS[arm] / "postprocess.py", OUT_DIR / ("postprocess_%s.py" % arm))
FINDINGS["_meta"] = dict(notebook_id=NOTEBOOK_ID, run_id=RUN_ID,
                         generated_for="handoff to Claude - do not edit by hand",
                         artifacts=sorted(p.name for p in OUT_DIR.iterdir()) + ["dataset_findings.json"])
(OUT_DIR / "dataset_findings.json").write_text(
    json.dumps(FINDINGS, ensure_ascii=False, indent=1, sort_keys=True, default=str), encoding="utf-8")
print("\nartefak ->", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()): print("   %-26s %8.1f KB" % (p.name, p.stat().st_size / 1e3))

dump_handoff()

print("\n%d/%d checks passed" % (n_pass, len(CHECKS)))
from IPython.display import FileLink, display
for arm, a in ARMS.items():
    if not a["ship"]:
        print("arm %s  %-38s %s" % (arm, a["what"], "cocok dengan run tersimpan, pipeline tidak berubah"
              if own[arm] else "!!! TIDAK COCOK dengan run tersimpan. Kirim JSON ke Claude."))
        continue
    ZP = FINDINGS["zip." + arm]
    print("arm %s  %-38s %s" % (arm, a["what"], ">>> SIAP SUBMIT" if READY[arm] else
                                ">>> JANGAN SUBMIT. Kirim blok JSON di atas ke Claude."))
    print("        %s  (%.2f GB, sha256[:16] %s)" % (ZP["path"], ZP["size_gb"], ZP["sha16"]))
    display(FileLink(os.path.relpath(ZP["path"])))
print("\nunduh: panel kanan -> Output -> <nama zip> -> titik tiga -> Download, atau link di atas.")


## Setelah notebook ini

1. Unzip output ke root repo (`findings/08-packaging-tts-all/` + ZIP-nya).
2. Kalau L bertanda **SIAP SUBMIT**, unduh ZIP-nya dan cocokkan `sha256[:16]` dengan cetakan sel 11.
3. Smoke test dulu (tidak memotong kuota), lalu submission penuh.
4. Baca skor test L terhadap **J 0,2655 / F 0,2660**. Dev tidak meramal dimensi ini.
